# 04 - Vocabulary, Subword Tokenization: BPE, WordPiece & Unigram

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain the problems with **word-level** and **character-level** tokenization.
- Implement **Byte-Pair Encoding (BPE)** from scratch and understand how GPT-family models tokenize text.
- Explain **WordPiece** (likelihood-based merging) as used by BERT.
- Describe **Unigram/SentencePiece** as a probabilistic alternative.
- Demonstrate how **subword tokenization handles OOV** (out-of-vocabulary) words.

## Prerequisites

- Completed Notebooks 01-03 (text preprocessing, BoW/TF-IDF, stemming/lemmatization).
- Basic Python (dictionaries, string operations, sorting).
- Understanding of word frequency and vocabulary concepts.

## Table of Contents

1. [The Tokenization Problem](#1-the-tokenization-problem)
2. [Word-Level Tokenization](#2-word-level-tokenization)
3. [Character-Level Tokenization](#3-character-level-tokenization)
4. [Subword Tokenization: The Sweet Spot](#4-subword-tokenization-the-sweet-spot)
5. [BPE: Byte-Pair Encoding](#5-bpe-byte-pair-encoding)
6. [WordPiece: Likelihood-Based Merging](#6-wordpiece-likelihood-based-merging)
7. [Unigram / SentencePiece (Brief)](#7-unigram--sentencepiece-brief)
8. [Code: Implement Simplified BPE](#8-code-implement-simplified-bpe)
9. [Code: Tokenization Examples](#9-code-tokenization-examples)
10. [Code: OOV Handling with Subword](#10-code-oov-handling-with-subword)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)
12. [Exercises](#12-exercises)
13. [Exercise Solutions](#13-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import re
import numpy as np
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Set

print("Setup complete.")

## 1. The Tokenization Problem

Every NLP model must convert text into numerical tokens. The choice of tokenization strategy has profound effects:

| Strategy | Vocab Size | Sequence Length | OOV Handling | Semantics |
|----------|:----------:|:---------------:|:------------:|:---------:|
| **Word-level** | Very large (100K+) | Short | Poor (UNK) | Good per token |
| **Character-level** | Very small (~100) | Very long | Perfect | Poor per token |
| **Subword** | Medium (30K-50K) | Medium | Good | Good balance |

Modern models (GPT, BERT, T5) all use **subword tokenization**.

## 2. Word-Level Tokenization

Split text at whitespace and punctuation boundaries. Each unique word gets an ID.

### Problems

- **Huge vocabulary**: English has 170,000+ words in current use; add technical terms, names, typos, and you easily exceed 1M.
- **OOV (Out-of-Vocabulary)**: Any word not seen during training becomes `<UNK>`.
  - `"transformerify"` -> `<UNK>` (never seen this made-up word)
  - `"COVID-19"` -> `<UNK>` (not in pre-2020 vocabularies)
- **Morphological blindness**: `"walk"`, `"walks"`, `"walked"`, `"walking"` are four separate tokens with no shared representation.
- **Multilingual difficulty**: Adding more languages multiplies vocabulary size.

In [ ]:
# ---- Word-level tokenization problems ----

def word_tokenize(text: str) -> List[str]:
    """Simple word-level tokenizer."""
    return re.findall(r"\w+|[^\w\s]", text.lower())

# Build vocabulary from training corpus
train_corpus = [
    "The cat sat on the mat.",
    "The dog played in the park.",
    "I love natural language processing.",
    "Machine learning is fascinating.",
]

vocab = set()
for text in train_corpus:
    vocab.update(word_tokenize(text))

print(f"Training vocabulary: {sorted(vocab)}")
print(f"Vocabulary size: {len(vocab)}")
print()

# Now try to tokenize unseen text
test_text = "The transformer model is groundbreaking for NLP."
test_tokens = word_tokenize(test_text)
oov_tokens = [t for t in test_tokens if t not in vocab]

print(f"Test text: {test_text}")
print(f"Tokens:    {test_tokens}")
print(f"OOV:       {oov_tokens}")
print(f"OOV rate:  {len(oov_tokens)/len(test_tokens)*100:.1f}%")
print()
print("Problem: 'transformer', 'model', 'groundbreaking', 'nlp' are all <UNK>!")

## 3. Character-Level Tokenization

Use individual characters as tokens. Vocabulary is tiny (26 letters + punctuation + digits).

### Problems

- **Very long sequences**: `"transformer"` = 11 tokens instead of 1.
- **Lost word-level semantics**: The model must learn to compose characters into meaningful units.
- **Slower training**: Longer sequences mean more computation (especially for attention-based models where cost is $O(n^2)$).
- **Harder to learn**: Composing `t-r-a-n-s-f-o-r-m-e-r` into the concept of "transformer" is a lot of work.

In [ ]:
# ---- Character-level tokenization ----

def char_tokenize(text: str) -> List[str]:
    """Character-level tokenizer."""
    return list(text.lower())

text = "The transformer model"
word_tokens = word_tokenize(text)
char_tokens = char_tokenize(text)

print(f"Text: {text!r}")
print(f"Word tokens ({len(word_tokens):2d}): {word_tokens}")
print(f"Char tokens ({len(char_tokens):2d}): {char_tokens}")
print()
print(f"Sequence length ratio: {len(char_tokens)/len(word_tokens):.1f}x longer!")
print()

# Character vocabulary is tiny
char_vocab = set(char_tokenize(" ".join(train_corpus)))
print(f"Character vocabulary size: {len(char_vocab)}")
print(f"Characters: {sorted(char_vocab)}")
print()
print("Advantage: Zero OOV! Any text can be represented.")
print("Disadvantage: Very long sequences lose word-level meaning.")

## 4. Subword Tokenization: The Sweet Spot

**Key insight:** Split rare words into common subword units while keeping frequent words whole.

- `"transformer"` -> `["transform", "er"]` (both common subwords)
- `"unhappiness"` -> `["un", "happiness"]` or `["un", "happi", "ness"]`
- `"the"` -> `["the"]` (frequent, kept whole)

### Three Main Algorithms

| Algorithm | Strategy | Used By |
|-----------|----------|--------|
| **BPE** | Bottom-up: merge most frequent byte pairs | GPT-2, GPT-3, GPT-4, RoBERTa |
| **WordPiece** | Bottom-up: merge pairs that maximize likelihood | BERT, DistilBERT, Electra |
| **Unigram** | Top-down: start large, prune least useful | T5, ALBERT, XLNet (via SentencePiece) |

## 5. BPE: Byte-Pair Encoding

### Algorithm

1. **Initialize**: Start with character-level vocabulary + end-of-word marker.
2. **Count pairs**: Count frequency of all adjacent symbol pairs across the corpus.
3. **Merge best**: Merge the most frequent pair into a new symbol.
4. **Repeat**: Go to step 2. Stop after $k$ merges (hyperparameter).

### Example

Corpus word frequencies: `{"low": 5, "lower": 2, "newest": 6, "widest": 3}`

```
Initial:  l o w _ (5), l o w e r _ (2), n e w e s t _ (6), w i d e s t _ (3)
Step 1:   Count pairs -> (e, s) is most frequent (6+3=9) -> merge to "es"
Step 2:   Count pairs -> (es, t) is most frequent (6+3=9) -> merge to "est"
Step 3:   Count pairs -> (l, o) is most frequent (5+2=7) -> merge to "lo"
... and so on
```

### Properties

- **Deterministic**: Given a corpus, BPE always produces the same merges.
- **Frequency-driven**: Common subwords get their own tokens.
- **No linguistic knowledge**: Purely statistical, works for any language.

## 6. WordPiece: Likelihood-Based Merging

WordPiece is similar to BPE but uses a different merge criterion.

### Difference from BPE

Instead of merging the most **frequent** pair, WordPiece merges the pair that **maximizes the likelihood** of the training data:

$$\text{score}(x, y) = \frac{\text{freq}(xy)}{\text{freq}(x) \times \text{freq}(y)}$$

This favors merging pairs where the combination is much more common than you would expect by chance.

### WordPiece Notation

- Continuation tokens are marked with `##` prefix:
  - `"unhappiness"` -> `["un", "##happi", "##ness"]`
  - `"playing"` -> `["play", "##ing"]`
- First subword of each word has no prefix.

### BERT Vocabulary

- ~30,000 WordPiece tokens
- Includes full words (common ones) and subwords (prefixed with `##`)
- Special tokens: `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`, `[UNK]`

## 7. Unigram / SentencePiece (Brief)

### Unigram Model

Unlike BPE (bottom-up merging), Unigram works **top-down**:

1. **Initialize** with a very large vocabulary (e.g., all substrings up to length $k$).
2. **Estimate** probabilities for each subword using EM (Expectation-Maximization).
3. **Prune**: Remove subwords whose removal causes the smallest increase in loss.
4. **Repeat** until target vocabulary size is reached.

The tokenization of a word is the segmentation that **maximizes the probability**:

$$P(\mathbf{x}) = \prod_{i=1}^{M} P(x_i)$$

where $x_1, \ldots, x_M$ is a segmentation of word $\mathbf{x}$ into $M$ subwords.

### SentencePiece

- A **library** (not an algorithm) that implements both BPE and Unigram.
- Treats the input as a raw character stream (no pre-tokenization needed).
- Language-agnostic: works directly on Unicode text.
- Used by T5, ALBERT, XLNet, mBART.

### Comparison

| Aspect | BPE | WordPiece | Unigram |
|--------|-----|-----------|--------|
| Direction | Bottom-up | Bottom-up | Top-down |
| Merge criterion | Frequency | Likelihood | Probability |
| Tokenization | Deterministic | Deterministic | Probabilistic (MAP) |
| Subword marker | None (or `\u0120`) | `##` prefix | `\u2581` (SentencePiece) |

## 8. Code: Implement Simplified BPE

We implement BPE from scratch to understand the merge process.

In [ ]:
class SimpleBPE:
    """Simplified Byte-Pair Encoding implementation.
    
    This is an educational implementation that demonstrates the core BPE
    algorithm. Production implementations (e.g., in HuggingFace tokenizers)
    are heavily optimized.
    """
    
    def __init__(self, num_merges: int = 20):
        self.num_merges = num_merges
        self.merges: List[Tuple[str, str]] = []  # ordered list of merge rules
        self.vocab: Set[str] = set()
    
    def _get_word_freqs(self, corpus: List[str]) -> Dict[tuple, int]:
        """Convert corpus to word frequencies with character-level splits.
        
        Each word becomes a tuple of characters + end-of-word marker '</w>'.
        """
        word_freqs = Counter()
        for text in corpus:
            for word in text.lower().split():
                # Split into characters, add end-of-word marker
                chars = tuple(list(word) + ["</w>"])
                word_freqs[chars] += 1
        return dict(word_freqs)
    
    def _get_pair_freqs(self, word_freqs: Dict[tuple, int]) -> Counter:
        """Count frequencies of all adjacent symbol pairs."""
        pair_freqs = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_freqs[pair] += freq
        return pair_freqs
    
    def _merge_pair(self, word_freqs: Dict[tuple, int],
                    pair: Tuple[str, str]) -> Dict[tuple, int]:
        """Merge all occurrences of a pair in the word frequency dict."""
        new_word_freqs = {}
        merged = pair[0] + pair[1]
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                # Try to match the pair
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def fit(self, corpus: List[str]) -> 'SimpleBPE':
        """Learn BPE merge rules from the corpus."""
        word_freqs = self._get_word_freqs(corpus)
        
        # Initialize vocabulary with all characters
        self.vocab = set()
        for word in word_freqs:
            self.vocab.update(word)
        
        print(f"Initial vocab ({len(self.vocab)} tokens): {sorted(self.vocab)}")
        print()
        
        self.merges = []
        for step in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(word_freqs)
            if not pair_freqs:
                break
            
            # Find most frequent pair
            best_pair = pair_freqs.most_common(1)[0]
            pair, freq = best_pair
            
            # Merge
            word_freqs = self._merge_pair(word_freqs, pair)
            merged_token = pair[0] + pair[1]
            self.merges.append(pair)
            self.vocab.add(merged_token)
            
            print(f"Merge {step+1:2d}: ('{pair[0]}', '{pair[1]}') -> '{merged_token}'  "
                  f"(freq={freq})")
        
        print(f"\nFinal vocab ({len(self.vocab)} tokens): {sorted(self.vocab)}")
        return self
    
    def tokenize(self, word: str) -> List[str]:
        """Tokenize a single word using learned merge rules."""
        # Start with characters + end-of-word marker
        tokens = list(word.lower()) + ["</w>"]
        
        # Apply merges in order
        for pair in self.merges:
            merged = pair[0] + pair[1]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (i < len(tokens) - 1
                    and tokens[i] == pair[0]
                    and tokens[i + 1] == pair[1]):
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        
        return tokens
    
    def tokenize_text(self, text: str) -> List[str]:
        """Tokenize a full text string."""
        all_tokens = []
        for word in text.split():
            all_tokens.extend(self.tokenize(word))
        return all_tokens

In [ ]:
# ---- Train BPE on a small corpus ----

corpus = [
    "low low low low low",
    "lower lower",
    "newest newest newest newest newest newest",
    "widest widest widest",
    "new new new new new new new new",
]

bpe = SimpleBPE(num_merges=15)
bpe.fit(corpus)

## 9. Code: Tokenization Examples

Let us see how our trained BPE tokenizes various words.

In [ ]:
# ---- Tokenize words with trained BPE ----

test_words = ["low", "lower", "lowest", "new", "newest", "newer", "wide", "widest"]

print("BPE Tokenization Results:")
print(f"{'Word':<12} {'Tokens':<35} {'# Tokens'}")
print("-" * 55)
for word in test_words:
    tokens = bpe.tokenize(word)
    print(f"{word:<12} {str(tokens):<35} {len(tokens)}")

In [ ]:
# ---- Tokenize full sentences ----

sentences = [
    "the newest model is the lowest",
    "a new lower estimate",
    "the widest new discovery",
]

print("Full Sentence Tokenization:")
print("=" * 60)
for sent in sentences:
    tokens = bpe.tokenize_text(sent)
    print(f"  Input:  {sent}")
    print(f"  Tokens: {tokens}")
    print(f"  Count:  {len(tokens)}")
    print()

## 10. Code: OOV Handling with Subword

The key advantage of subword tokenization: **no word is truly OOV** because any word can be decomposed into characters (worst case) or known subwords.

In [ ]:
# ---- Demonstrate OOV handling ----

# Words NOT seen in training
oov_words = ["slower", "newest", "lowest", "snowed", "tower", "news"]

print("OOV Handling Comparison:")
print("=" * 70)
print(f"{'Word':<12} {'In Train Vocab?':<18} {'Word-Level':<15} {'BPE Subwords'}")
print("-" * 70)

# Word-level vocabulary from training
train_word_vocab = set()
for text in corpus:
    train_word_vocab.update(text.lower().split())

for word in oov_words:
    in_vocab = "Yes" if word in train_word_vocab else "No"
    word_level = word if word in train_word_vocab else "<UNK>"
    bpe_tokens = bpe.tokenize(word)
    print(f"{word:<12} {in_vocab:<18} {word_level:<15} {bpe_tokens}")

print()
print("Key insight: BPE decomposes unknown words into known subwords.")
print("Even totally new words are handled by falling back to characters.")

In [ ]:
# ---- Vocabulary size comparison ----

# Build a larger corpus to illustrate vocabulary growth
large_corpus = [
    "the cat sat on the mat and the dog sat on the log",
    "natural language processing is a field of artificial intelligence",
    "machine learning algorithms learn patterns from data",
    "deep learning models use neural networks with many layers",
    "transformers revolutionized natural language understanding",
    "word embeddings capture semantic relationships between words",
    "attention mechanisms allow models to focus on relevant parts",
    "pre training on large corpora improves downstream task performance",
    "tokenization is the first step in any text processing pipeline",
    "subword tokenization handles out of vocabulary words gracefully",
]

# Word-level vocabulary
word_vocab = set()
for text in large_corpus:
    word_vocab.update(text.lower().split())

# Character vocabulary
char_vocab = set()
for text in large_corpus:
    char_vocab.update(text.lower())

# BPE vocabulary
bpe_large = SimpleBPE(num_merges=30)
print("Training BPE on larger corpus:")
print("=" * 60)
bpe_large.fit(large_corpus)

print()
print("Vocabulary Size Comparison:")
print(f"  Character-level: {len(char_vocab)} tokens")
print(f"  BPE (30 merges): {len(bpe_large.vocab)} tokens")
print(f"  Word-level:      {len(word_vocab)} tokens")

In [ ]:
# ---- Simulated WordPiece comparison ----
# We show how WordPiece would tokenize the same text
# (using ## prefix notation like BERT)

def simulate_wordpiece(word: str, known_subwords: set) -> List[str]:
    """Simulate WordPiece greedy longest-match tokenization.
    
    This is a simplified version. Real WordPiece uses the trained
    vocabulary from likelihood-based merging.
    """
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        found = False
        while start < end:
            substr = word[start:end]
            if start > 0:
                substr = "##" + substr
            if substr in known_subwords:
                tokens.append(substr)
                found = True
                break
            end -= 1
        if not found:
            # Fall back to single character
            token = word[start]
            if start > 0:
                token = "##" + token
            tokens.append(token)
            start += 1
        else:
            start = end
    return tokens

# Simulated BERT-like subword vocabulary
wp_vocab = {
    "transform", "##er", "##s", "##ing", "##ed", "##ation",
    "un", "##happy", "##ness", "play", "re",
    "the", "is", "a", "new", "low", "##est",
    "##a", "##b", "##c", "##d", "##e", "##f", "##g", "##h",
    "##i", "##j", "##k", "##l", "##m", "##n", "##o", "##p",
    "##q", "##r", "##t", "##u", "##v", "##w", "##x", "##y", "##z",
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l",
    "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z",
}

wp_test_words = ["transformer", "transformers", "transforming",
                 "unhappiness", "playing", "replaying", "newest"]

print("Simulated WordPiece Tokenization (BERT-style):")
print(f"{'Word':<18} {'WordPiece Tokens'}")
print("-" * 50)
for word in wp_test_words:
    tokens = simulate_wordpiece(word, wp_vocab)
    print(f"{word:<18} {tokens}")

## 11. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using word-level tokenization for rare domains | High OOV rate, many `<UNK>` tokens | Use BPE or WordPiece |
| Too few BPE merges | Very fragmented tokens, long sequences | Increase num_merges (typical: 30K-50K) |
| Too many BPE merges | Approaches word-level, loses subword benefit | Reduce merges or use vocabulary cap |
| Pre-tokenizing before SentencePiece | SentencePiece expects raw text | Feed raw text to SentencePiece |
| Applying text normalization after subword tokenization | Corrupts learned subword tokens | Always normalize **before** tokenization |
| Different tokenizer for train vs inference | Token IDs mismatch, garbage output | Always use the **same** tokenizer |
| Ignoring special tokens (`[CLS]`, `[SEP]`, `<s>`, `</s>`) | Model misinterprets input boundaries | Include required special tokens |
| Assuming BPE merges are linguistic | They are purely statistical | Don't read linguistic meaning into subword splits |

**Debugging tip:** When a model produces garbage output, first check that you are using the **exact same tokenizer** that was used during training. Even minor version differences can change the vocabulary.

## 12. Exercises

### Exercise 1: Implement BPE on a Small Corpus

Train BPE on the given corpus and answer the questions.

In [ ]:
exercise_corpus = [
    "hug hug hug hug",
    "pug pug pug",
    "hugs hugs",
    "pugs pugs",
    "bug bug bug bug bug",
    "bugs bugs bugs",
    "mug mug",
    "mugs",
    "hugging hugging",
]

# TODO:
# 1. Train BPE with 10 merges on this corpus
# 2. What is the first merge? Why?
# 3. Tokenize the word "plugs" (not in training data)
# 4. How many tokens does "hugging" produce?
# Your code here

### Exercise 2: Vocabulary Size vs Sequence Length Trade-off

Train BPE with different numbers of merges and observe the trade-off between vocabulary size and average sequence length.

In [ ]:
tradeoff_corpus = [
    "natural language processing is fascinating",
    "machine learning algorithms learn from data",
    "deep learning uses neural networks",
    "transformers changed natural language understanding",
    "attention mechanisms are powerful tools",
]

test_sentence = "natural language learning is transforming understanding"

# TODO:
# 1. Train BPE with num_merges = [0, 5, 10, 20, 40]
# 2. For each, tokenize the test_sentence
# 3. Record vocabulary size and token count
# 4. Plot or print the trade-off
# Your code here

## 13. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----

print("Exercise 1: BPE on hug/pug/bug corpus")
print("=" * 60)

ex_bpe = SimpleBPE(num_merges=10)
ex_bpe.fit(exercise_corpus)

print()
print("Q: What is the first merge? Why?")
print(f"A: The first merge is {ex_bpe.merges[0]}")
print("   because 'ug' is the most frequent adjacent pair across")
print("   hug, pug, bug, mug, hugs, pugs, bugs, mugs, hugging.")
print()

# Tokenize OOV word
plugs_tokens = ex_bpe.tokenize("plugs")
print(f"Q: Tokenize 'plugs': {plugs_tokens}")
print("   Even though 'plugs' was never in training, BPE handles it.")
print()

hugging_tokens = ex_bpe.tokenize("hugging")
print(f"Q: Tokenize 'hugging': {hugging_tokens}")
print(f"   Number of tokens: {len(hugging_tokens)}")

In [ ]:
# ---- Exercise 2 Solution ----

print("Exercise 2: Vocabulary Size vs Sequence Length")
print("=" * 60)

merge_counts = [0, 5, 10, 20, 40]
results = []

for n_merges in merge_counts:
    bpe_ex = SimpleBPE(num_merges=n_merges)
    # Suppress output for cleaner display
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        bpe_ex.fit(tradeoff_corpus)
    
    tokens = bpe_ex.tokenize_text(test_sentence)
    results.append((n_merges, len(bpe_ex.vocab), len(tokens)))

print(f"\n{'Merges':<10} {'Vocab Size':<15} {'Token Count':<15} {'Tokens'}")
print("-" * 70)
for n_merges, vocab_size, token_count in results:
    # Re-tokenize for display
    bpe_ex = SimpleBPE(num_merges=n_merges)
    with contextlib.redirect_stdout(io.StringIO()):
        bpe_ex.fit(tradeoff_corpus)
    tokens = bpe_ex.tokenize_text(test_sentence)
    tokens_str = str(tokens)[:50] + "..." if len(str(tokens)) > 50 else str(tokens)
    print(f"{n_merges:<10} {vocab_size:<15} {token_count:<15} {tokens_str}")

print()
print("Observation: More merges -> larger vocab, shorter sequences.")
print("This is the fundamental trade-off in subword tokenization.")